<a href="https://colab.research.google.com/github/krishnasaijoga/OpenEnvHackathon/blob/main/Mod_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5: Train a Wordle Agent with GRPO

Fine-tune Qwen3-1.7B to play Wordle using GRPO (Group Relative Policy Optimization) via TRL and OpenEnv.

**Time:** ~90 min (training) · **Difficulty:** Advanced · **GPU:** A100 required (Colab Pro or similar)

Based on the [TRL OpenEnv Wordle example](https://github.com/huggingface/trl/blob/main/examples/notebooks/openenv_wordle_grpo.ipynb).

In [1]:
    # Install dependencies
!uv pip install -Uq "git+https://github.com/huggingface/trl.git@v0.29.1" "openenv-core==0.2.2" trackio "vllm>=0.11.0" bitsandbytes "git+https://github.com/meta-pytorch/OpenEnv.git@v0.2.2#subdirectory=envs/textarena_env"


In [ ]:
# Log in to Hugging Face (required for model access and pushing results)
from huggingface_hub import notebook_login
notebook_login()

## 1. Initialize the Environment

Connect to the TextArena Wordle environment hosted on HF Spaces.

> **For production use:** Duplicate the Space to your own account to avoid concurrency limits.

In [ ]:
from textarena_env import TextArenaEnv

textarena_url = "https://https://mroyme-textarena-env.hf.space"  # Duplicate and update this!
env = TextArenaEnv(base_url=textarena_url)
print(f"Connected to: {textarena_url}")

Connected to: https://https://mroyme-textarena-env.hf.space


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 2. Init Model and Tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print(f"Model: {model_name}")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Model: Qwen/Qwen3-1.7B


## 3. Define the System Prompt

In [ ]:
system_prompt = """
You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

## GAME RULES

1. The target is a 5-letter English word
2. You have 6 attempts to guess the correct word
3. After each guess, you receive color-coded feedback:
   - GREEN: Letter is correct and in the correct position
   - YELLOW: Letter is in the word but in the wrong position
   - GRAY: Letter is not in the word at all
4. All guesses must be valid 5-letter English words
5. You cannot reuse a word you've already guessed

## RESPONSE FORMAT

Only respond with your next guess in square brackets, e.g., [crane].

## STRATEGIC APPROACH

Do not repeat the same guess twice.

### Opening Strategy
- Start with words rich in common vowels (A, E, I, O, U) and consonants (R, S, T, L, N)
- Optimal starters: CRANE, SLATE, STARE, AROSE, IRATE

### Mid-Game Strategy
- Use confirmed GREEN letters in their correct positions
- Place YELLOW letters in different positions than where they appeared
- Eliminate GRAY letters from consideration

## YOUR GOAL

Solve the Wordle in as few guesses as possible.
"""

## 4. Helper Functions

In [ ]:
def make_user_prompt(prompt_text, messages):
    """Build a structured prompt from game state and message history."""
    history = format_history(messages)
    prompt_section = prompt_text.strip() if prompt_text.strip() else "Wordle-v0"
    history_section = history if history else "[PROMPT] Awaiting first feedback."
    return (
        f"Game prompt:\n{prompt_section}\n\n"
        f"Conversation so far:\n{history_section}\n\n"
        "Reply with your next guess enclosed in square brackets."
    )


def format_history(messages):
    """Format message history with category tags."""
    lines = []
    for message in messages:
        tag = message.category or "MESSAGE"
        content = message.content.strip()
        if content:
            lines.append(f"[{tag}] {content}")
    return "\n".join(lines)


def scale_repetition_score(previous_occurrences, max_occurrences):
    """Scale repetition penalty: 1.0 = novel guess, 0.0 = fully repeated."""
    if max_occurrences == 0:
        return 0.0
    return (max_occurrences - previous_occurrences) / max_occurrences


print("Helper functions defined.")

Helper functions defined.


## 5. Define the Rollout Function

The rollout function plays one full Wordle game per prompt. It's called by `GRPOTrainer` during training.

In [ ]:
from collections import defaultdict
from textarena_env import TextArenaAction
from textarena_env.rewards import extract_feedback_counts, extract_guess, extract_wordle_feedback
from trl.experimental.openenv import generate_rollout_completions


def rollout_once(trainer, env, tokenizer, dataset_prompt, system_prompt, max_turns):
    """Execute one full Wordle episode."""
    result = env.reset()
    observation = result.observation

    prompt_ids = []
    completion_ids = []
    logprobs = []
    green_scores = []
    yellow_scores = []
    repetition_scores = []
    correct_scores = []
    guess_counts = defaultdict(int)

    for _turn in range(max_turns):
        if result.done:
            break

        base_prompt = observation.prompt or dataset_prompt
        user_prompt = make_user_prompt(base_prompt, observation.messages)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )

        rollout_outputs = generate_rollout_completions(trainer, [prompt_text])[0]
        prompt_ids.extend(rollout_outputs["prompt_ids"])
        completion_ids.extend(rollout_outputs["completion_ids"])
        logprobs.extend(rollout_outputs["logprobs"])
        completion_text = rollout_outputs.get("text") or tokenizer.decode(
            rollout_outputs["completion_ids"], skip_special_tokens=True
        )

        guess = extract_guess(completion_text)
        result = env.step(TextArenaAction(message=guess))
        observation = result.observation
        correct_score = float(result.reward or 0.0)
        feedback = extract_wordle_feedback(observation)

        previous_occurrences = guess_counts[guess]
        repetition_score = scale_repetition_score(previous_occurrences, len(guess_counts))
        guess_counts[guess] += 1

        if not feedback:
            green_score, yellow_score = 0.0, 0.0
        else:
            green_count, yellow_count = extract_feedback_counts(feedback)
            green_score = green_count / 5.0
            yellow_score = yellow_count / 5.0

        repetition_scores.append(repetition_score)
        green_scores.append(green_score)
        yellow_scores.append(yellow_score)
        correct_scores.append(correct_score)

    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "correct_reward": correct_scores[-1] if correct_scores else 0.0,
        "green_reward": green_scores[-1] if green_scores else 0.0,
        "yellow_reward": yellow_scores[-1] if yellow_scores else 0.0,
        "repetition_reward": repetition_scores[-1] if repetition_scores else 0.0,
    }


def rollout_func(prompts, trainer=None):
    """Rollout function called by GRPOTrainer."""
    episode_prompt_ids = []
    episode_completion_ids = []
    episode_logprobs = []
    correctness_rewards = []
    green_rewards = []
    yellow_rewards = []
    repetition_rewards = []

    for prompt_text in prompts:
        episode = rollout_once(
            trainer=trainer,
            env=env,
            tokenizer=tokenizer,
            dataset_prompt=prompt_text,
            system_prompt=system_prompt,
            max_turns=6,
        )
        episode_prompt_ids.append(episode["prompt_ids"])
        episode_completion_ids.append(episode["completion_ids"])
        episode_logprobs.append(episode["logprobs"])
        correctness_rewards.append(episode["correct_reward"])
        green_rewards.append(episode["green_reward"])
        yellow_rewards.append(episode["yellow_reward"])
        repetition_rewards.append(episode["repetition_reward"])

    return {
        "prompt_ids": episode_prompt_ids,
        "completion_ids": episode_completion_ids,
        "logprobs": episode_logprobs,
        "correct_reward": correctness_rewards,
        "green_reward": green_rewards,
        "yellow_reward": yellow_rewards,
        "repetition_reward": repetition_rewards,
    }


print("Rollout functions defined.")

/tmp/ipykernel_7661/2898770919.py:4: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.openenv import generate_rollout_completions


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/usr/local/lib/python3.12/dist-packages/trl/experimental/openenv/utils.py:24: UserWarning: TRL currently supports vLLM versions: 0.10.2, 0.11.0, 0.11.1, 0.11.2, 0.12.0. You have version 0.18.0 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for remova

Rollout functions defined.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 6. Define Reward Functions

Four reward signals for richer gradient information.

In [ ]:
def reward_correct(completions, **kwargs):
    rewards = kwargs.get("correct_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

def reward_greens(completions, **kwargs):
    rewards = kwargs.get("green_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

def reward_yellows(completions, **kwargs):
    rewards = kwargs.get("yellow_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

def reward_repetition(completions, **kwargs):
    rewards = kwargs.get("repetition_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

print("Reward functions: correct, greens, yellows, repetition")

Reward functions: correct, greens, yellows, repetition


## 7. Create Dataset

In [ ]:
from datasets import Dataset

dataset_size = 1000
dataset = Dataset.from_dict({"prompt": ["Play Wordle like an expert."] * dataset_size})
print(f"Dataset: {len(dataset)} prompts")

Dataset: 1000 prompts


## 8. Configure GRPO Training

In [ ]:
from trl import GRPOConfig

output_dir = "wordle-grpo-Qwen3-1.7B"

grpo_config = GRPOConfig(
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=64,
    per_device_train_batch_size=1,
    warmup_steps=20,
    num_generations=2,
    max_completion_length=8,
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.1,
    vllm_max_model_length=2048,
    output_dir=output_dir,
    report_to="trackio",
    trackio_space_id=output_dir,
    logging_steps=1,
    save_steps=10,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    push_to_hub=True,
)

print(f"Output: {output_dir}")
print(f"vLLM mode: colocate (generation + training on same GPU)")

Output: wordle-grpo-Qwen3-1.7B
vLLM mode: colocate (generation + training on same GPU)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 9. Create Trainer and Train

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model_name,
    processing_class=tokenizer,
    reward_funcs=[
        reward_correct,
        reward_greens,
        reward_yellows,
        reward_repetition,
    ],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

/usr/local/lib/python3.12/dist-packages/trl/generation/__init__.py:22: UserWarning: TRL currently supports vLLM versions: 0.10.2, 0.11.0, 0.11.1, 0.11.2, 0.12.0. You have version 0.18.0 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/usr/local/lib/python3.12/dist-packages/trl/generation/vllm_client.py:39: UserWarning: TRL currently supports vLLM versions: 0.10.2, 0.11.0, 0.11.1, 0.11.2, 0.12.0. You have version 0.18.0 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/usr/local/lib/python3.12/dist-packages/trl/generation/vllm_generation.py:37: UserWarning: TRL currently supports vLLM versions: 0.10.2, 0.11.0, 0.11.1, 0.11.2, 0.12.0. You have version 0.18.0 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datet

NameError: name 'reward_correct' is not defined

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Check GPU before training
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name} — {max_memory} GB total, {start_gpu_memory} GB reserved")

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# Train (~90 minutes on A100)
trainer_stats = trainer.train()

In [ ]:
# Memory stats after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_for_training = round(used_memory - start_gpu_memory, 3)

print(f"Training time: {round(trainer_stats.metrics['train_runtime']/60, 1)} minutes")
print(f"Peak memory: {used_memory} GB ({round(used_memory/max_memory*100, 1)}% of {max_memory} GB)")
print(f"Memory for training: {used_for_training} GB")

## 10. Save and Push

In [ ]:
env.close()
trainer.save_model(output_dir)
trainer.push_to_hub()
print(f"Model saved to {output_dir} and pushed to Hub.")

## 11. Evaluate: Play Wordle with the Trained Model

In [ ]:
from transformers import AutoModelForCausalLM
from envs.textarena_env.rewards import extract_guess

# Load the fine-tuned model (replace with your HF username)
fine_tuned_model = AutoModelForCausalLM.from_pretrained(
    output_dir, dtype="auto", device_map="auto"
)


def play_wordle(env, model, tokenizer, max_turns=6):
    """Play one game and print the results."""
    result = env.reset()
    observation = result.observation
    print(f"Prompt: {observation.prompt[:100]}...")

    for turn in range(max_turns):
        if result.done:
            break

        user_prompt = make_user_prompt(observation.prompt, observation.messages)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True,
            tokenize=False, enable_thinking=False,
        )

        model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=512)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
        guess = extract_guess(generated_text)

        print(f"\nTurn {turn + 1}: {guess}")

        result = env.step(TextArenaAction(message=guess))
        observation = result.observation
        for msg in observation.messages:
            print(f"  [{msg.category}] {msg.content}")

    print(f"\nResult: reward={result.reward}, done={result.done}")


# Play a game
eval_env = TextArenaEnv(base_url=textarena_url)
try:
    play_wordle(eval_env, fine_tuned_model, tokenizer)
finally:
    eval_env.close()

## Summary

What you did:
1. Connected to the TextArena Wordle environment via OpenEnv
2. Defined a system prompt, rollout function, and 4 reward signals
3. Trained Qwen3-1.7B with GRPO for ~90 minutes on an A100
4. Evaluated the trained model on live Wordle games

The key insight: **OpenEnv makes the environment a plug-in.** Swap Wordle for any other OpenEnv environment — your Module 4 word game, a coding environment, a math problem — and the training pipeline stays the same.

### What's next

- **Improve the model:** Longer training, larger models, better reward shaping
- **Build your own environment:** Use Module 4's pattern, plug it into this pipeline
- **Scale up:** See the [Scaling appendix](../README.md#bonus-scaling-openenv) for multi-container deployment
- **Explore the Hub:** Browse [openenv environments](https://huggingface.co/collections/openenv/environment-hub) for inspiration